# Reconnaissance faciale avec YOLOv8

Les modèles de deep learning apprennent des représentations hiérarchiques à partir des données brutes. Les **CNN** appliquent des filtres appris spatialement — détectant les contours, puis les textures, puis les structures de haut niveau. <br><br>ResNet, par exemple, empile des blocs CNN résiduels pour classifier les images de manière fiable.<br><br> Les **EfficientNets** font évoluer les CNN en profondeur, en largeur et en résolution via un coefficient composé, atteignant une meilleure précision par paramètre EfficientNet-B0 égale ResNet-50 avec environ 5× moins de paramètres.
<br><br>
Ici, nous affinons **YOLOv8**, qui repose sur un backbone CNN, pour résoudre une tâche de classification binaire : *owner* vs *not_owner*.

In [1]:
!pip install ultralytics gradio

  Using cached gradio-6.12.0-py3-none-any.whl.metadata (17 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached gradio_client-2.4.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.0-py3-none-any.whl.metadata (428 bytes)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached gradio-6.12.0-py3-none-any.whl (19.6 MB)
Using cached gradio_client-2.4.1-py3-none-any.whl (59 kB)
Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl (30 kB)
Using cached groovy-0.1.2-py3-none-any.whl (14 kB)
Using cached hf_gradio

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'd:\\python\\lib\\site-packages\\_brotli.cp313-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



## 1. Imports

In [1]:
import os
import cv2
import uuid
import shutil
import random
from ultralytics import YOLO
import time
import ipywidgets as widgets
import threading
from IPython.display import display as ipy_display

## 2. Collecte des images

Un classifieur a besoin d'exemples étiquetés. Nous capturons deux classes :
- **owner** — des photos de vous
- **not_owner** — des photos de n'importe qui d'autre, ou des arrière-plans variés

YOLOv8 classification attend la structure de dossiers suivante :

```
data/dataset/
├── train/
│   ├── owner/
│   └── not_owner/
└── val/
    ├── owner/
    └── not_owner/
```

Nous collectons d'abord les images brutes, puis les divisons automatiquement. Visez au minimum 40 à 50 images par classe.

Appuyez sur **S** pour sauvegarder une image, **Q** pour quitter.

In [2]:
RAW_PATH = os.path.join('data', 'raw')
CLASSES = ['owner', 'not_owner']
NUM_IMAGES = 40  # par classe

for cls in CLASSES:
    os.makedirs(os.path.join(RAW_PATH, cls), exist_ok=True)

print("Dossiers prêts :", os.listdir(RAW_PATH))

Dossiers prêts : ['not_owner', 'owner']


In [7]:
def capturer_images(cls):
    cap = cv2.VideoCapture(2)
    count = 0
    running = True
    last_frame = None

    img_widget = widgets.Image(format='jpeg', width=640)
    save_btn   = widgets.Button(description='📸 Sauvegarder', button_style='success')
    quit_btn   = widgets.Button(description='❌ Quitter',      button_style='danger')
    status     = widgets.Label(value=f"0/{NUM_IMAGES} sauvegardées — classe : {cls}")

    def on_save(b):
        nonlocal count, last_frame
        if last_frame is not None:
            path = os.path.join(RAW_PATH, cls, f"{cls}_{uuid.uuid1()}.jpg")
            cv2.imwrite(path, last_frame.copy())
            count += 1
            status.value = f"{count}/{NUM_IMAGES} sauvegardées — classe : {cls}"
            if count >= NUM_IMAGES:
                on_quit(None)

    def on_quit(b):
        nonlocal running
        running = False

    save_btn.on_click(on_save)
    quit_btn.on_click(on_quit)

    ipy_display(widgets.VBox([img_widget, widgets.HBox([save_btn, quit_btn]), status]))

    def apercu():
        nonlocal last_frame
        while running and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            last_frame = frame
            annotated = frame.copy()
            cv2.putText(annotated, f"{cls}  {count}/{NUM_IMAGES}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            _, buf = cv2.imencode('.jpg', annotated)
            img_widget.value = buf.tobytes()
            time.sleep(0.05)

        cap.release()
        status.value = f"✅ Terminé — {count}/{NUM_IMAGES} sauvegardées pour '{cls}'"

    thread = threading.Thread(target=apercu, daemon=True)
    thread.start()
    return thread

In [8]:
# Capture des images OWNER — placez-vous face à la caméra
t = capturer_images('owner')

In [9]:
t.join()

In [10]:
# Capture des images NOT_OWNER — demandez à quelqu'un d'autre de se placer devant la caméra, ou éloignez-vous
t = capturer_images('not_owner')

In [11]:
t.join()

## 3. Construction du jeu de données

Nous divisons chaque classe 80/20 entre entraînement et validation. Le modèle apprend sur la partition d'entraînement ; la partition de validation mesure sa capacité à généraliser sur des images qu'il n'a jamais vues — ce qui compte en pratique.

In [3]:
DATASET_PATH = os.path.join('data', 'dataset')
VAL_SPLIT = 0.2

for cls in CLASSES:
    src = os.path.join(RAW_PATH, cls)
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    cut = int(len(images) * (1 - VAL_SPLIT))
    splits = {'train': images[:cut], 'val': images[cut:]}

    for split, files in splits.items():
        dest = os.path.join(DATASET_PATH, split, cls)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(src, f), os.path.join(dest, f))

    print(f"{cls}: {cut} train / {len(images) - cut} val")

print("\nDataset ready at:", DATASET_PATH)

owner: 44 train / 12 val
not_owner: 42 train / 11 val

Dataset ready at: data\dataset


## 4. Affinage de YOLOv8

Plutôt qu'entraîner depuis zéro, nous chargeons `yolov8n-cls.pt` — un modèle nano pré-entraîné sur ImageNet. Nous remplaçons et ré-entraînons uniquement sa tête de classification sur nos deux classes. C'est le **transfer learning** : le backbone sait déjà extraire des caractéristiques visuelles ; il suffit de lui apprendre à reconnaître votre visage spécifique. La convergence est rapide et fonctionne bien même avec peu d'images.

In [ ]:
model = YOLO('yolov8n-cls.pt')
model.train(
    data=DATASET_PATH,
    epochs=7,
    imgsz=64,
    batch=4,
    name='face_recognition',
    patience=5,
    workers=0,
    device='cpu',
)

# Chercher best.pt directement sur le disque
import glob
weights = glob.glob('runs/classify/face_recognition*/weights/best.pt')
best = max(weights, key=os.path.getmtime)
print("Meilleurs poids sauvegardés dans :", best)

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.232  Python-3.13.5 torch-2.9.1+cpu CPU (Intel Core i5-8365U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=7, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=64, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=face_recognition2, nbs=64, nms=False, opset=None, op

In [ ]:
# model = YOLO('yolov8n-cls.pt')

# model.train(
#     data=DATASET_PATH,
#     epochs=30,
#     imgsz=224,
#     batch=16,
#     name='face_recognition',
#     patience=10,   # arrêt anticipé si la perte de validation ne s'améliore plus
# )

# print("Meilleurs poids sauvegardés dans :", model.trainer.best)

## 5. Test sur une image fixe

Avant de brancher le modèle dans l'application en direct, vérifiez qu'il prédit correctement sur un échantillon de validation.

In [4]:
best_model = YOLO(os.path.join('runs', 'classify', 'face_recognition', 'weights', 'best.pt'))

val_owner_dir = os.path.join(DATASET_PATH, 'val', 'owner')
sample = os.path.join(val_owner_dir, os.listdir(val_owner_dir)[0])

results = best_model(sample)
probs = results[0].probs
print("Classe prédite :", results[0].names[probs.top1])
print(f"Confiance : {probs.top1conf:.2%}")


image 1/1 D:\Centrale\Centrale Tech\workshop_2025_2026\Data-AI\Workshop_5\data\dataset\val\owner\owner_ab95882e-39ca-11f1-8093-a4b1c18c5063.jpg: 64x64 not_owner 0.61, owner 0.39, 7.2ms
Speed: 5.7ms preprocess, 7.2ms inference, 0.1ms postprocess per image at shape (1, 3, 64, 64)
Classe prédite : not_owner
Confiance : 60.72%


## 6. Test en direct avec la webcam

Une dernière vérification avant le déploiement : faites tourner le modèle en temps réel sur le flux de la webcam. Vert = owner, rouge = not_owner. Appuyez sur **Q** pour quitter.

In [5]:
def tester_modele_live(model_path, camera_index=2):
    best_model = YOLO(model_path)
    cap = cv2.VideoCapture(camera_index)

    img_widget = widgets.Image(format='jpeg', width=640)
    quit_btn   = widgets.Button(description='Quitter', button_style='danger')
    status     = widgets.Label(value="Modèle en cours d'exécution...")
    running    = True
    last_frame = None

    def on_quit(b):
        nonlocal running
        running = False

    quit_btn.on_click(on_quit)
    ipy_display(widgets.VBox([img_widget, quit_btn, status]))

    def apercu():
        while running and cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            results    = best_model(frame, verbose=False)
            probs      = results[0].probs
            label      = results[0].names[probs.top1]
            conf       = probs.top1conf.item()
            color      = (0, 200, 80) if label == 'owner' else (0, 60, 220)
            cv2.putText(frame, f"{label}  {conf:.0%}", (12, 42),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
            _, buf = cv2.imencode('.jpg', frame)
            img_widget.value = buf.tobytes()
            status.value = f"{'Owner' if label == 'owner' else 'Not Owner'}  —  confiance : {conf:.0%}"
            time.sleep(0.05)

        cap.release()
        status.value = "⏹️ Caméra arrêtée."

    thread = threading.Thread(target=apercu, daemon=True)
    thread.start()
    return thread

In [6]:
tester_modele_live(r'D:\Centrale\Centrale Tech\workshop_2025_2026\Data-AI\Workshop_5\runs\classify\face_recognition\weights\best.pt', camera_index=2)

<Thread(Thread-4 (apercu), started daemon 13380)>

# 6. Tester son modèle avec Gradio

In [7]:
def lancer_interface_gradio(model_path):
    import gradio as gr

    best_model = YOLO(model_path)

    def predire(image):
        results = best_model(image, verbose=False)
        probs   = results[0].probs
        label   = results[0].names[probs.top1]
        conf    = probs.top1conf.item()
        return {name: float(probs.data[i]) for i, name in results[0].names.items()}

    interface = gr.Interface(
        fn=predire,
        inputs=gr.Image(type='numpy', label="Importer une image"),
        outputs=gr.Label(num_top_classes=2, label="Résultat"),
        title="Face Recognition — Owner / Not Owner",
        description="Importez une image pour tester le modèle entraîné. Vert = owner, rouge = not_owner.",
        flagging_mode="never",
    )
    interface.launch(share=False)

In [3]:
lancer_interface_gradio(best_model)